In [ ]:
import zipfile
import os

zip_path = "/content/bbc.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/bbc")

In [ ]:
!find /content/bbc -type d

In [ ]:
os.listdir("/content/bbc")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/bbc/bbc-text.csv")
df.head()


In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

In [ ]:
stop_words = set(stopwords.words("english"))
lemm = WordNetLemmatizer()

In [ ]:
def preprocess(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # 3. Remove numbers and punctuation
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # 4. Tokenize
    tokens = word_tokenize(text)

    # 5. Remove stopwords + keep meaningful words
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    # 6. Lemmatize words
    tokens = [lemm.lemmatize(w) for w in tokens]

    # 7. Remove extremely short words
    tokens = [w for w in tokens if len(w) > 2]

    # 8. Reconstruct text
    return " ".join(tokens)


In [ ]:
df["clean_text"] = df["text"].apply(preprocess)
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,      # limit to top 5000 informative words
    min_df=2,               # ignore extremely rare words
    max_df=0.85,            # ignore extremely common words
    ngram_range=(1,2)       # use unigrams + bigrams
)

X_tfidf = tfidf.fit_transform(df["clean_text"])

print("TF-IDF shape:", X_tfidf.shape)


In [ ]:
from sklearn.cluster import KMeans

k = 5
kmeans_tfidf = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_tfidf = kmeans_tfidf.fit_predict(X_tfidf)

df["cluster_tfidf"] = labels_tfidf

df[["category", "cluster_tfidf"]].head()


In [ ]:
from sklearn.metrics import silhouette_score

sil_score = silhouette_score(X_tfidf, labels_tfidf)
print("Silhouette Score (TF-IDF KMeans):", sil_score)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

# Load a pretrained SBERT model (well-balanced and efficient)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Prepare the documents from the preprocessed text
documents = df['clean_text'].tolist()

# Encode the documents
embeddings = model.encode(documents, show_progress_bar=True, convert_to_numpy=True)

print("Embedding shape:", embeddings.shape)

In [ ]:
from sklearn.cluster import KMeans

k = 5
kmeans_sbert = KMeans(n_clusters=k, random_state=42)
clusters_sbert = kmeans_sbert.fit_predict(embeddings)

df['cluster_sbert'] = clusters_sbert

df[['category', 'cluster_sbert']]

In [ ]:
from sklearn.metrics import silhouette_score

score_sbert = silhouette_score(embeddings, clusters_sbert)
print("Silhouette Score (SBERT KMeans):", score_sbert)

In [ ]:


scores = {}
inertias = {}
K = range(2, 10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(embeddings)
    inertias[k] = kmeans.inertia_
    scores[k] = silhouette_score(embeddings, labels)

print("Silhouette Scores:", scores)
print("Inertia:", inertias)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(list(scores.keys()), list(scores.values()), marker='o')
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs k (SBERT)")
plt.show()

In [ ]:
!pip install umap-learn

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# UMAP reduction to 2D
umap_reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

umap_2d = umap_reducer.fit_transform(embeddings)

# Fit KMeans with best k=5
kmeans_sbert = KMeans(n_clusters=5, random_state=42)
clusters_sbert = kmeans_sbert.fit_predict(embeddings)

# Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=umap_2d[:, 0],
    y=umap_2d[:, 1],
    hue=clusters_sbert,
    palette='tab10',
    s=40
)

plt.title("UMAP Visualization of SBERT Embeddings + KMeans Clusters")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.legend(title="Cluster")
plt.show()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(documents)

feature_names = np.array(tfidf.get_feature_names_out())

# For each cluster → find top keywords
def get_top_keywords(cluster_id, tfidf_matrix, labels, top_n=10):
    cluster_docs = tfidf_matrix[labels == cluster_id]
    mean_tfidf = cluster_docs.mean(axis=0).A1
    top_indices = mean_tfidf.argsort()[::-1][:top_n]
    return feature_names[top_indices]

# Print keywords for each cluster
for c in range(5):  # 5 clusters
    print(f"\nTop Keywords for Cluster {c}:")
    print(get_top_keywords(c, tfidf_matrix, clusters_sbert, top_n=12))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query, top_n=5):
    # Encode query using SBERT
    query_emb = model.encode([query], convert_to_numpy=True)

    # Compute cosine similarity
    sims = cosine_similarity(query_emb, embeddings)[0]

    # Top N results
    top_idx = sims.argsort()[::-1][:top_n]

    print("\nQuery:", query)
    print("\nTop Results:\n")

    for i in top_idx:
        print(f"Score: {sims[i]:.4f}")
        print(f"Cluster: {clusters_sbert[i]}")
        print(f"Text: {documents[i][:200]}...")
        print("-" * 80)

# Test the search engine
semantic_search("latest technology updates", top_n=5)


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Candidate human topic labels (BBC style)
possible_labels = [
    "sports", "business", "technology",
    "politics", "entertainment"
]

# Encode possible labels
label_embeddings = model.encode(possible_labels)

cluster_names = {}

for c in range(5):
    # get top keywords for cluster c
    keywords = get_top_keywords(c, tfidf_matrix, clusters_sbert, top_n=8)
    keyword_text = " ".join(keywords)

    # encode keywords
    keyword_embed = model.encode([keyword_text])

    # similarity with all possible topic names
    sims = cosine_similarity(keyword_embed, label_embeddings)[0]

    # pick best label
    best_label = possible_labels[np.argmax(sims)]

    cluster_names[c] = best_label

print("\nAutomatic Cluster Labels:")
for c, label in cluster_names.items():
    print(f"Cluster {c} → {label}")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def search_engine(query):
    # Encode user query
    query_emb = model.encode([query], convert_to_numpy=True)

    # Compute cosine similarity
    sims = cosine_similarity(query_emb, embeddings)[0]

    # Top results
    top_idx = sims.argsort()[::-1][:5]

    results = []
    for i in top_idx:
        label = cluster_names[clusters_sbert[i]]
        text_preview = documents[i][:300].replace("\n", " ")
        score = round(float(sims[i]), 4)

        results.append(
            f"🔹 **Similarity:** {score}\n"
            f" **Cluster:** {label}\n"
            f" **Text:** {text_preview}...\n"
            "--------------------------------------"
        )

    return "\n\n".join(results)

    # Build the Gradio UI
interface = gr.Interface(
    fn=search_engine,
    inputs=gr.Textbox(lines=2, placeholder="Enter your search query..."),
    outputs="markdown",
    title="BBC News Semantic Search Engine",
    description="Search the BBC dataset using SBERT + KMeans Clustering",
)

interface.launch(debug=True)
